In [ ]:
import geopandas as gpd
import glob
import os
import rioxarray as rxr
import xarray as xr

## Settings

In [ ]:
PATH = "data/GHS_LAND_*.tif"
AOI = "aoi/A00_Granice_panstwa.shp"
RASTER_CHUNKS = "auto"
WATER_PIXEL_THRESHOLD = 95
NO_DATA_VALUE = 255
OUTPUT_DIRECTORY = "output"

## Open area of interest file

In [ ]:
gdf_aoi = gpd.read_file(AOI)
gdf_aoi

## Process raster files

In [ ]:
files = glob.glob(PATH)
files

In [ ]:
os.makedirs(OUTPUT_DIRECTORY, exist_ok=True)

In [ ]:
for file in files:
    print(f"Processing {file}...")
    with rxr.open_rasterio(file, cache=False, chunks=RASTER_CHUNKS) as src:
        
        print("Cropping to ROI...")
        cropped_image = src.rio.clip(gdf_aoi.geometry.values, crs=gdf_aoi.crs)

        print("Reclassifying values...")
        cropped_image_reclass = xr.where(cropped_image <= WATER_PIXEL_THRESHOLD, 1, NO_DATA_VALUE)

        print("Saving metadata...")
        cropped_image_reclass.rio.write_crs(cropped_image.rio.crs, inplace=True)

        print("Exporting output file...")
        output_file = os.path.basename(file).replace(".tif", f"_reclass_{WATER_PIXEL_THRESHOLD}.tif")
        output_path = os.path.join(OUTPUT_DIRECTORY, output_name)
        cropped_image_reclass.rio.to_raster(output_path, compress="lzw", dtype="uint8", nodata=NO_DATA_VALUE)
        
        print(f"Finished processing {file}\n")